# Model Training Analysis
This notebook trains the Conversion-Quality ML model interactively and displays its evaluation results.

In [19]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data import generate_data, get_train_val_test_splits
from src.model import engineer_features, MatchModel, evaluate_model
from src.baseline import BaselineRanker, evaluate_baseline


In [20]:
# Generate Synthetic Data
df = generate_data(num_samples=1000, random_state=42)
train_df, val_df, test_df = get_train_val_test_splits(df)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 600, Val: 200, Test: 200


In [21]:
# Engineer Features
X_train = engineer_features(train_df)
y_train = train_df['is_good_match']
X_val = engineer_features(val_df)
y_val = val_df['is_good_match']
X_test = engineer_features(test_df)
y_test = test_df['is_good_match']

X_train.head()

,overlap_ratio,missing_skills_count,seniority_gap,years_exp,job_seniority
24,0.4,3,4,6,2
467,0.4,3,5,6,1
539,0.5,2,5,9,4
531,0.6,2,3,8,5
618,0.8,1,1,3,2


In [22]:
# Baseline Evaluation
baseline = BaselineRanker(threshold=0.6)
baseline_metrics = evaluate_baseline(test_df, baseline, split_name="Test")

[Test Baseline] Precision: 0.738, Recall: 0.633, FPR: 0.073


In [23]:
# Train ML Model
model = MatchModel()
model.train(X_train, y_train)

print("Evaluating on Validation Set:")
evaluate_model(model, X_val, y_val, split_name="Val")

print("\nEvaluating on Test Set:")
model_metrics = evaluate_model(model, X_test, y_test, split_name="Test")

print(f"\nDelta vs Baseline Precision: {model_metrics['precision'] - baseline_metrics['precision']:.3f}")


Evaluating on Validation Set:
[Val Model] Precision: 0.867, Recall: 0.310, FPR: 0.013

Evaluating on Test Set:
[Test Model] Precision: 0.833, Recall: 0.204, FPR: 0.013

Delta vs Baseline Precision: 0.095


In [24]:
# Analyze Feature Importance (Coefficients)
import pandas as pd
coef_df = pd.DataFrame({
    'Feature': model.feature_names,
    'Coefficient': model.model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

coef_df

,Feature,Coefficient
0,overlap_ratio,2.544989
2,seniority_gap,0.084125
3,years_exp,-0.011345
4,job_seniority,-0.095470
1,missing_skills_count,-0.449326
